## Prepare questions from NACC for training GRPO

Created by: Sahana Kowshik

Date: 05/06/2025

In [1]:
import pandas as pd
import json
import random
import numpy as np

In [2]:
directory_path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc"

In [3]:
random_state = 42

In [4]:
train = pd.read_csv(f"{directory_path}/train.csv")

/scratch/ipykernel_963353/3940650821.py:1: DtypeWarning: Columns (20,22,24,26,28,40,43,45,47,50,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,164,175,178,188,216,219,221,223,225,227,229,231,233,235,237,239,241,243,245,247,249,251,253,255,257,259,261,263,265,267,269,271,396,398,400,420,422,431,444,453,492,571,599,607,679,693,696,699,716,727,733,808,819,823,825,831,892,947,948,949,957,958,959,960,1017,1022,1192,1196,1199,1395,1397,1399,1400,1402,1409,1411,1413,1414,1421,1423,1425,1427,1428,1435,1450,1464,1478,1492,1494,1530,1546,1548,1550,1552,1554,1556,1558,1560,1562,1564,1566,1568,1570,1572,1574,1576,1578,1580,1582,1584,1586,1588,1590,1592,1594,1596,1598,1600,1650,1651,1653,1654,1657,1658,1661,1662,1665,1666,1669,1670,1744,1803,1812,1814,1816,1818,1829,1831,1833,1841,1843,1845,1847,1855,1857,1859,1861,1887) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv(f"{directory_path}/train.csv")


In [5]:
# Add ETPR variables
def add_etpr_labels(row):
    value = row['NACCETPR']
    
    # NC
    if value == 88:
        row['ETPR'] = 'NC'
    
    # AD
    elif value == 1:
        row['ETPR'] = 'AD'

    # LBD
    elif value == 2:
        row['ETPR'] = 'LBD'

    # VD
    elif value == 8:
        row['ETPR'] = 'VD'

    # FTLD and its variants, including CBD and PSP, with or without ALS
    elif value in [4, 5, 6, 7]:
        row['ETPR'] = 'FTLD'
    
    # SEF: seizure, epilepsy, etc.
    elif value in [17, 18, 23, 26, 27, 28, 29]:
        row['ETPR'] = 'SEF'

    # Psychiatric conditions
    elif value in [19, 20, 21, 22, 24, 25]:
        row['ETPR'] = 'PSY'

    # Other degenerative
    elif value not in [30, 88, 99]:
        row['ETPR'] = 'ODE'

    else:
        row['ETPR'] = np.NaN

    return row


train = train.apply(add_etpr_labels, axis=1)

In [6]:
train['NACCUDSD'].value_counts()

NACCUDSD
1    17508
4    16725
3     8831
Name: count, dtype: int64

In [8]:
train['ETPR'].value_counts()

ETPR
NC      17508
AD      17102
FTLD     2238
LBD      1515
VD       1183
SEF       830
PSY       538
ODE       266
Name: count, dtype: int64

In [9]:
columns = ['ID', 'patient_summary', 'diag_summary', 'question', 'options', 'ground_truth', 'ground_truth_text', 'COHORT']

In [10]:
train["ID"] = train["NACCID"]
train.drop(['NACCID'], axis=1, inplace=True)

/scratch/ipykernel_963353/893815028.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train["ID"] = train["NACCID"]


## Neuropathology

In [11]:
# Function to add NP ground truth columns
def add_ground_truth(row):
    # Alzheimer's Disease
    if row['NPADNC'] in [2, 3]:
        row['NP_AD'] = 1
        row['NP_AVAIL'] = 1
    elif row['NPADNC'] in [0, 1]:
        row['NP_AD'] = 0
        row['NP_AVAIL'] = 1

    # Lewy Body Disease
    if row['NACCLEWY'] in [1, 2, 3, 4]:
        row['NP_LBD'] = 1
        row['NP_AVAIL'] = 1
    elif row['NACCLEWY'] == 0:
        row['NP_LBD'] = 0
        row['NP_AVAIL'] = 1

    # Frontotemporal Degeneration (FTLD-tau or other)
    if (row['NPFTDTAU'] == 1) | (row['NPFTDTDP'] == 1):
        row['NP_FTD'] = 1
        row['NP_AVAIL'] = 1
    elif (row['NPFTDTAU'] == 0) & (row['NPFTDTDP'] == 0):
        row['NP_FTD'] = 0
        row['NP_AVAIL'] = 1

    return row

In [12]:
# Apply the function across the training set
train = train.apply(add_ground_truth, axis=1)

# Check distribution of available neuropathology labels
train['NP_AVAIL'].value_counts()

NP_AVAIL
1.0    2000
Name: count, dtype: int64

In [39]:
# Keep only patients with neuropathology data available
train_np = train[(train['NP_AVAIL'] == 1)].reset_index(drop=True)
print(f"Length: {len(train_np)}")

Length: 2000


In [40]:
train_np[train_np['NP_LBD'].isna()]

,ABRUPT,ABUSOTHR,ABUSX,ADGCEXOM,ADGCEXR,ADGCGWAS,ADGCRND,AFIBRILL,AFRAID,AGIT,...,WAIS,WEIGHT,WHITEVOL,WMHVOL,WONDRFUL,WRTHLESS,diag_summary,patient_summary,sort_key,sort_year


In [41]:
train_np['VISITGAP'].value_counts()

VISITGAP
1.0    806
0.0    556
2.0    374
3.0    264
Name: count, dtype: int64

In [42]:
print("NC")
print(f"Maximum difference between year of death and visit year {train_np[train_np['NACCUDSD'] == 1]['VISITGAP'].max()}")
print(f"Minimum difference between year of death and visit year {train_np[train_np['NACCUDSD'] == 1]['VISITGAP'].min()}")
print(f"Mean difference between year of death and visit year {train_np[train_np['NACCUDSD'] == 1]['VISITGAP'].mean()}")

NC
Maximum difference between year of death and visit year nan
Minimum difference between year of death and visit year nan
Mean difference between year of death and visit year nan


In [43]:
print("MCI")
print(f"Maximum difference between year of death and visit year {train_np[train_np['NACCUDSD'] == 3]['VISITGAP'].max()}")
print(f"Minimum difference between year of death and visit year {train_np[train_np['NACCUDSD'] == 3]['VISITGAP'].min()}")
print(f"Mean difference between year of death and visit year {train_np[train_np['NACCUDSD'] == 3]['VISITGAP'].mean()}")

MCI
Maximum difference between year of death and visit year 3.0
Minimum difference between year of death and visit year 0.0
Mean difference between year of death and visit year 1.2548076923076923


In [44]:
print("DE")
print(f"Maximum difference between year of death and visit year {train_np[train_np['NACCUDSD'] == 4]['VISITGAP'].max()}")
print(f"Minimum difference between year of death and visit year {train_np[train_np['NACCUDSD'] == 4]['VISITGAP'].min()}")
print(f"Mean difference between year of death and visit year {train_np[train_np['NACCUDSD'] == 4]['VISITGAP'].mean()}")

DE
Maximum difference between year of death and visit year 3.0
Minimum difference between year of death and visit year 0.0
Mean difference between year of death and visit year 1.1635044642857142


In [45]:
# print(f"Overlap with previous data: {len(set(prev_train_with_questions['ID']).intersection(set(train_np['ID'])))}")

In [46]:
import random, string
import pandas as pd

# Neuropathology Question Variants
np_question_variants_one = [
    "Which of the following pathology is causing the patient's cognitive impairment based on the clinical information provided?",
    "Using the provided clinical information, determine the pathological cause of the patient's cognitive decline from the options listed below.",
    "Based on the clinical data, identify the neuropathology responsible for the patient's cognitive impairment by selecting from the given choices.",
    "From the options below, select the disease pathology causing cognitive impairment as indicated by the clinical information.",
    "Review the provided clinical information and choose the pathology underlying the patient's cognitive symptoms from the available options."
]

np_question_variants_mixed = [
    "Which of the following pathologies are causing the patient's cognitive impairment based on the clinical information provided?",
    "Using the provided clinical information, determine the pathological causes of the patient's cognitive decline from the options listed below.",
    "Based on the clinical data, identify the neuropathologies responsible for the patient's cognitive impairment by selecting from the given choices.",
    "From the options below, select the disease pathologies causing cognitive impairment as indicated by the clinical information.",
    "Review the provided clinical information and choose the pathologies underlying the patient's cognitive symptoms from the available options."
]

# Pull the bare answer texts into a list – easier to shuffle
NP_OPTION_TEXTS_ONE = [
    "Alzheimer's disease pathology (AD)",
    "Lewy body pathology (LBD)",
    "Frontotemporal Lobar Degeneration with tau pathology or TDP-43 pathology (FTLD)",
    "No listed option is correct",
]

NP_OPTION_TEXTS_MIXED = [
    "Alzheimer's disease pathology (AD) and Lewy body pathology (LBD)",
    "Alzheimer's disease pathology (AD) and Frontotemporal Lobar Degeneration with tau pathology or TDP-43 pathology (FTLD)",
    "Lewy body pathology (LBD) and Frontotemporal Lobar Degeneration with tau pathology or TDP-43 pathology (FTLD)",
    "Alzheimer's disease pathology (AD), Lewy body pathology (LBD) and Frontotemporal Lobar Degeneration with tau pathology or TDP-43 pathology (FTLD)",
    "No listed option is correct",
]

# map each neuropathology pattern → the **text** of the correct option
NP_ANSWER_MAP = {
    (1, 1, 1): "Alzheimer's disease pathology (AD), Lewy body pathology (LBD) and Frontotemporal Lobar Degeneration with tau pathology or TDP-43 pathology (FTLD)",
    (0, 1, 1): "Lewy body pathology (LBD) and Frontotemporal Lobar Degeneration with tau pathology or TDP-43 pathology (FTLD)",
    (1, 0, 1): "Alzheimer's disease pathology (AD) and Frontotemporal Lobar Degeneration with tau pathology or TDP-43 pathology (FTLD)",
    (1, 1, 0): "Alzheimer's disease pathology (AD) and Lewy body pathology (LBD)",
    (0, 0, 1): "Frontotemporal Lobar Degeneration with tau pathology or TDP-43 pathology (FTLD) only",
    (0, 1, 0): "Lewy body pathology (LBD) only",
    (1, 0, 0): "Alzheimer's disease pathology (AD) only",
    (0, 0, 0): "No listed option is correct",
}




In [ ]:
def add_np_question(row, *, NP_OPTION_TEXTS, np_question_variants, seed=None):
    """
    Adds three new columns to each row:
    ─ question : one of the phrasing variants           (string)
    ─ options  : the shuffled, letter-prefixed choices  (multiline string)
    ─ ground_truth : the **letter** (A-H) of the correct choice (string)
    """
    rng = random.Random(seed if seed is not None else row.name)  # reproducible per-row if you like

    # 1️⃣ Choose a question at random
    row["question"] = rng.choice(np_question_variants)

    # 2️⃣ Shuffle the options and assign fresh letters
    shuffled = NP_OPTION_TEXTS.copy()
    rng.shuffle(shuffled)
    letters = list(string.ascii_uppercase[: len(shuffled)])
    row["options"] = "\n".join(f"{ltr}. {opt}" for ltr, opt in zip(letters, shuffled))


    # 4️⃣ Convert that to the new letter after shuffling
    correct_letter = letters[shuffled.index(row['separation_identifier'].replace(" only", ""))]
    row["ground_truth"] = correct_letter
    
    row['ground_truth_text'] = row['separation_identifier'].replace(" only", "")

    return row

def split_np_cases(df):
    
    df['separation_identifier'] = df.apply(lambda row: NP_ANSWER_MAP[(row["NP_AD"], row["NP_LBD"], row["NP_FTD"])], axis=1)

    # Select cases based on the ground_truth_text column, not the options column
    one_pathology = df[df['separation_identifier'].str.contains("only", na=False)].reset_index(drop=True)
    mixed_pathology = df[df['separation_identifier'].str.contains("and", na=False)].reset_index(drop=True)
    no_pathology = df[df['separation_identifier'] == "No listed option is correct"].reset_index(drop=True)

    # Split no_pathology cases between one_pathology and mixed_pathology
    half = len(no_pathology) // 2
    one_pathology = pd.concat([one_pathology, no_pathology.iloc[:half]]).sample(frac=1, random_state=random_state).reset_index(drop=True)
    mixed_pathology = pd.concat([mixed_pathology, no_pathology.iloc[half:]]).sample(frac=1, random_state=random_state).reset_index(drop=True)

    # Apply add_np_question with the correct option sets
    one_pathology = one_pathology.apply(add_np_question, NP_OPTION_TEXTS=NP_OPTION_TEXTS_ONE, np_question_variants=np_question_variants_one, axis=1)
    mixed_pathology = mixed_pathology.apply(add_np_question, NP_OPTION_TEXTS=NP_OPTION_TEXTS_MIXED, np_question_variants=np_question_variants_mixed, axis=1)
    
    assert len(set(one_pathology['ID']).intersection(set(mixed_pathology['ID']))) == 0
    
    return  pd.concat([one_pathology, mixed_pathology]).sample(frac=1, random_state=random_state).reset_index(drop=True)

# apply to your neuropath-labelled dataframe
train_np = split_np_cases(train_np)

In [57]:
index = -106
print(train_np.iloc[index]['question'])
print(train_np.iloc[index]['options'])
print(train_np.iloc[index]['ground_truth'])
print(train_np.iloc[index]['NP_AD'])
print(train_np.iloc[index]['NP_LBD'])
print(train_np.iloc[index]['NP_FTD'])

Which of the following pathology is causing the patient's cognitive impairment based on the clinical information provided?
A. No listed option is correct
B. Lewy body pathology (LBD)
C. Alzheimer's disease pathology (AD)
D. Frontotemporal Lobar Degeneration with tau pathology or TDP-43 pathology (FTLD)
C
1.0
0.0
0.0


In [53]:
train_np['ground_truth_text'].value_counts()

ground_truth_text
Alzheimer's disease pathology (AD)                                                                                                                   633
Alzheimer's disease pathology (AD) and Lewy body pathology (LBD)                                                                                     585
Frontotemporal Lobar Degeneration with tau pathology or TDP-43 pathology (FTLD)                                                                      299
No listed option is correct                                                                                                                          165
Lewy body pathology (LBD)                                                                                                                             97
Alzheimer's disease pathology (AD) and Frontotemporal Lobar Degeneration with tau pathology or TDP-43 pathology (FTLD)                                92
Alzheimer's disease pathology (AD), Lewy body pathology (LBD) an

In [58]:
# Select final columns to keep
train_np = train_np[columns]  # Make sure 'columns' is defined somewhere earlier

In [59]:
train_np['Q_TYPE'] = 'Neuropath'

In [60]:
# Update set of NACCIDs used to avoid duplication later
np_ids = set(train_np['ID'])
len(np_ids)

2000

## Datscan

In [61]:
train[(~train['ID'].isin(np_ids)) & (train['DATSCAN'].isin([0,1]))]['DATSCAN'].value_counts()

DATSCAN
0.0    311
1.0    151
Name: count, dtype: int64

In [62]:
# Filter training data: exclude patients already used in NP task and keep only those with valid amyloid PET results
train_dat = (
    train
    .loc[
        (~train['ID'].isin(np_ids)) &
        (train['DATSCAN'].isin([0, 1]))
    ]
    .groupby('DATSCAN', group_keys=False)  # Ensure balanced sampling by amyloid status
    .apply(lambda x: x.sample(n=min(len(x), 100), random_state=42))
    # .sample(frac=1, random_state=42)  # Shuffle all rows
    .reset_index(drop=True)
)

train_dat['DATSCAN'].value_counts()

/scratch/ipykernel_963353/476130558.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(len(x), 100), random_state=42))


DATSCAN
0.0    100
1.0    100
Name: count, dtype: int64

In [63]:
import random, string, json

# Keys related to DATScan that should be removed from the patient summary
dat_keys_remove = [
    'Dopamine transporter scan (DATscan) evidence for Lewy body disease'
]

dat_question_variants = [
    "Is there evidence of Lewy body disease in the brain based on the provided information?",
    "Does the provided information suggest the presence of Lewy body disease in the brain?",
    "Is the patient likely to have Lewy body pathology in the brain based on the available data?",
    "Based on the information given, is there evidence consistent with Lewy body disease in the brain?",
    "Does the information support the presence of underlying Lewy body pathology in the brain?"
]

# Raw answer texts
DAT_OPTION_TEXTS = ["Yes", "No"]

# Map DATSCAN value to answer text
DAT_ANSWER_MAP = {
    1: "Yes",  # DATScan positive
    0: "No",   # DATScan negative
}

def add_dat_question(row, *, seed=None):
    rng = random.Random(seed if seed is not None else row.name)

    # 1️⃣ Randomly assign a question variant
    row['question'] = rng.choice(dat_question_variants)

    # 2️⃣ Remove sensitive DATScan key from patient summary
    json_file = json.loads(row['patient_summary'])
    for key in dat_keys_remove:
        if key in json_file.get('Imaging evidence', {}):
            json_file['Imaging evidence'].pop(key)
    row['patient_summary'] = json.dumps(json_file, indent=4)

    # 3️⃣ Shuffle options
    shuffled = DAT_OPTION_TEXTS.copy()
    rng.shuffle(shuffled)
    letters = list(string.ascii_uppercase[:len(shuffled)])
    row["options"] = "\n".join(f"{ltr}. {opt}" for ltr, opt in zip(letters, shuffled))

    # 4️⃣ Determine correct letter based on DATSCAN value
    correct_text = DAT_ANSWER_MAP[row['DATSCAN']]
    correct_letter = letters[shuffled.index(correct_text)]
    
    row["ground_truth"] = correct_letter
    row['ground_truth_text'] = correct_text

    return row


In [64]:
# Apply the function across the DAT training dataset
train_dat = train_dat.apply(add_dat_question, axis=1)

In [65]:
# print("Checking if datscan is accidentally mentioned in train_dat cases")
# for i, row in train_dat.iterrows():
#     if (row['DATSCAN'] in [0,1]) & ('Dopamine transporter scan (DATscan) evidence for Lewy body disease' in row['patient_summary']):
#         print(i, "ERROR available in patient_summary")

# print("Checking if datscan is accidentally mentioned in previous train_pet summaries")
# x = prev_train_summary[(prev_train_summary['ID'].isin(prev_train_dat['ID'])) & (prev_train_summary['Q_TYPE'] == 'DATSCAN')]
# print(len(x))
# for i, row in x.iterrows():
#     if 'Dopamine transporter scan (DATscan) evidence for Lewy body disease' in row['patient_summary']:
#         print(i, "ERROR available in patient_summary")

#     if 'datscan' in row['visit_summary'].lower():
#         print(i, "ERROR available in visit_summary")

# print("Checking if datscan is not present in other cases")
# x = train[(~train['ID'].isin(train_dat['ID']))]
# print(len(x))
# for i, row in x.iterrows():
#     if (row['DATSCAN'] in [0,1]) & ('Dopamine transporter scan (DATscan) evidence for Lewy body disease' not in row['patient_summary']):
#         print(i, "ERROR NOT available in patient_summary")

In [66]:
# Check how many times each question variant was selected
print(train_dat['question'].value_counts())
# Check distribution of DATScan labels
print(train_dat['NACCUDSD'].value_counts())
print(train_dat['ground_truth'].value_counts())
print(train_dat['ground_truth_text'].value_counts())

question
Does the information support the presence of underlying Lewy body pathology in the brain?            43
Does the provided information suggest the presence of Lewy body disease in the brain?                42
Based on the information given, is there evidence consistent with Lewy body disease in the brain?    41
Is there evidence of Lewy body disease in the brain based on the provided information?               41
Is the patient likely to have Lewy body pathology in the brain based on the available data?          33
Name: count, dtype: int64
NACCUDSD
4    105
3     63
1     32
Name: count, dtype: int64
ground_truth
B    102
A     98
Name: count, dtype: int64
ground_truth_text
No     100
Yes    100
Name: count, dtype: int64


In [67]:
index = -60
print(train_dat.iloc[index]['options'])
print(train_dat.iloc[index]['ground_truth'])
print(train_dat.iloc[index]['DATSCAN'])

A. No
B. Yes
B
1.0


In [68]:
# Select only the necessary columns
train_dat = train_dat[columns]

In [69]:
# Update set of NACCIDs used to avoid reusing subjects in other tasks
dat_ids = set(train_dat['ID'])
len(dat_ids)

200

In [70]:
train_dat['Q_TYPE'] = 'DATSCAN'

In [71]:
for key in dat_keys_remove:
    for i, row in train_dat.iterrows():
        if key in row['patient_summary']:
            print(i, key)
            # break

## Amylpet

In [72]:
train[(~train['ID'].isin(np_ids.union(dat_ids))) & (train['AMYLPET'].isin([0,1]))]['AMYLPET'].value_counts()

AMYLPET
1.0    1716
0.0    1538
Name: count, dtype: int64

In [73]:
# Filter training data: exclude patients already used in NP task and keep only those with valid amyloid PET results
train_pet = (
    train
    .loc[
        (~train['ID'].isin(np_ids.union(dat_ids))) &
        (train['AMYLPET'].isin([0, 1]))
    ]
    .groupby('AMYLPET', group_keys=False)  # Ensure balanced sampling by amyloid status
    .apply(lambda x: x.sample(n=min(len(x), 1000), random_state=42))
    # .sample(frac=1, random_state=42)  # Shuffle all rows
    .reset_index(drop=True)
)


/scratch/ipykernel_963353/2525435841.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(len(x), 1000), random_state=42))


In [74]:
train_pet['AMYLPET'].value_counts()

AMYLPET
0.0    1000
1.0    1000
Name: count, dtype: int64

In [75]:
import random, string, json

# Keys related to amyloid biomarkers that should be removed from the patient summary
pet_keys_remove = [
    'Abnormally elevated amyloid on PET',
    'Abnormally low amyloid in CSF',
    'FDG-PET pattern of AD',
    'Tau PET evidence for AD',
    'Abnormally elevated CSF Tau or pTau'
]

pet_question_variants = [
    "Is the patient likely to have elevated amyloid pathology in the brain given the provided information?",
    "Does the patient likely have abnormal amyloid accumulation in the brain based on the clinical data?",
    "Based on the available information, is the patient likely amyloid positive in the brain?",
    "Does the evidence suggest the presence of elevated amyloid burden in the brain in this patient?",
    "Is there a high likelihood of underlying amyloid pathology in the brain based on the patient's profile?"
]


# Raw answer texts
PET_OPTION_TEXTS = ["Yes", "No"]

# Map AMYLPET value to answer text
PET_ANSWER_MAP = {
    1: "Yes",  # Amyloid positive
    0: "No",   # Amyloid negative
}

def add_pet_question(row, *, seed=None):
    rng = random.Random(seed if seed is not None else row.name)

    # 1️⃣ Choose a question variant randomly
    row['question'] = rng.choice(pet_question_variants)

    # 2️⃣ Remove sensitive fields from the summary
    json_file = json.loads(row['patient_summary'])
    for key in pet_keys_remove:
        if key in json_file.get('Imaging evidence', {}):
            json_file['Imaging evidence'].pop(key)
        if key in json_file.get('CSF evidence', {}):
            json_file['CSF evidence'].pop(key)
    row['patient_summary'] = json.dumps(json_file, indent=4)

    # 3️⃣ Shuffle the answer options
    shuffled = PET_OPTION_TEXTS.copy()
    rng.shuffle(shuffled)
    letters = list(string.ascii_uppercase[:len(shuffled)])
    row["options"] = "\n".join(f"{ltr}. {opt}" for ltr, opt in zip(letters, shuffled))

    # 4️⃣ Determine correct letter based on AMYLPET value
    correct_text = PET_ANSWER_MAP[row['AMYLPET']]
    correct_letter = letters[shuffled.index(correct_text)]
    row["ground_truth"] = correct_letter
    
    row['ground_truth_text'] = correct_text

    return row


In [76]:
# Apply the function across the PET training dataset
train_pet = train_pet.apply(add_pet_question, axis=1)

In [77]:
# Check distribution of amyloid PET labels
print(train_pet['AMYLPET'].value_counts())

# Quick check: How many times each question variant was picked
print(train_pet['question'].value_counts())

print(train_pet['ground_truth'].value_counts())

print(train_pet['ground_truth_text'].value_counts())

AMYLPET
0.0    1000
1.0    1000
Name: count, dtype: int64
question
Is the patient likely to have elevated amyloid pathology in the brain given the provided information?      420
Does the evidence suggest the presence of elevated amyloid burden in the brain in this patient?            406
Is there a high likelihood of underlying amyloid pathology in the brain based on the patient's profile?    401
Does the patient likely have abnormal amyloid accumulation in the brain based on the clinical data?        390
Based on the available information, is the patient likely amyloid positive in the brain?                   383
Name: count, dtype: int64
ground_truth
B    1011
A     989
Name: count, dtype: int64
ground_truth_text
No     1000
Yes    1000
Name: count, dtype: int64


In [78]:
index = 0
print(train_pet.iloc[index]['options'])
print(train_pet.iloc[index]['ground_truth'])
print(train_pet.iloc[index]['ground_truth_text'])
print(train_pet.iloc[index]['AMYLPET'])

A. Yes
B. No
B
No
0.0


In [79]:
# Select only necessary columns
train_pet = train_pet[columns]

In [80]:
train_pet['Q_TYPE'] = 'Amyloid PET'

In [81]:
# print(train[train['NACCID'] == 'NACC867075'].iloc[0]['patient_summary'])

In [82]:
for key in pet_keys_remove:
    for i, row in train_pet.iterrows():
        if key in row['patient_summary']:
            print(i, key)
            # break

In [83]:
# Update set of NACCIDs used to avoid reusing subjects in other tasks
pet_ids = set(train_pet['ID'])
print(len(pet_ids))


2000


## Amylcsf

In [84]:
train[(~train['ID'].isin(np_ids.union(dat_ids).union(pet_ids))) & (train['AMYLCSF'].isin([0,1]))]['AMYLCSF'].value_counts()

AMYLCSF
1.0    607
0.0    498
Name: count, dtype: int64

In [85]:
# Filter training data: exclude patients already used in NP task and keep only those with valid amyloid PET results
train_csf = (
    train
    .loc[
        (~train['ID'].isin(np_ids.union(dat_ids).union(pet_ids))) & 
        (train['AMYLCSF'].isin([0, 1]))
    ]
    .groupby('AMYLCSF', group_keys=False)  # Ensure balanced sampling by amyloid status
    .apply(lambda x: x.sample(n=min(len(x), 400), random_state=42))
    .sample(frac=1, random_state=42)  # Shuffle all rows
    .reset_index(drop=True)
)

# Check the distribution of amyloid PET labels
train_csf['AMYLCSF'].value_counts()

/scratch/ipykernel_963353/987828470.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(len(x), 400), random_state=42))


AMYLCSF
1.0    400
0.0    400
Name: count, dtype: int64

In [86]:
train_csf[train_csf['AMYLCSF'] == 1]['AMYLPET'].value_counts()

AMYLPET
8.0    337
1.0     54
0.0      9
Name: count, dtype: int64

In [87]:
train[train['AMYLPET'] == 1]['AMYLCSF'].value_counts()

AMYLCSF
8.0    1523
1.0     243
0.0      66
Name: count, dtype: int64

In [88]:
import random, string, json

# Keys related to amyloid biomarkers that should be removed from the patient summary
csf_keys_remove = [
    'Abnormally elevated amyloid on PET',
    'Abnormally low amyloid in CSF',
    'FDG-PET pattern of AD',
    'Tau PET evidence for AD',
    'Abnormally elevated CSF Tau or pTau'
]

csf_question_variants = [
    "Is the patient likely to have elevated amyloid pathology in the brain given the provided information?",
    "Does the patient likely have abnormal amyloid accumulation in the brain based on the clinical data?",
    "Based on the available information, is the patient likely amyloid positive in the brain?",
    "Does the evidence suggest the presence of elevated amyloid burden in the brain in this patient?",
    "Is there a high likelihood of underlying amyloid pathology in the brain based on the patient's profile?"
]


# Raw answer texts
CSF_OPTION_TEXTS = ["Yes", "No"]

# Map AMYLPET value to answer text
CSF_ANSWER_MAP = {
    1: "Yes",  # Amyloid positive
    0: "No",   # Amyloid negative
}

def add_csf_question(row, *, seed=None):
    rng = random.Random(seed if seed is not None else row.name)

    # 1️⃣ Choose a question variant randomly
    row['question'] = rng.choice(csf_question_variants)

    # 2️⃣ Remove sensitive fields from the summary
    json_file = json.loads(row['patient_summary'])
    for key in csf_keys_remove:
        if key in json_file.get('Imaging evidence', {}):
            json_file['Imaging evidence'].pop(key)
        if key in json_file.get('CSF evidence', {}):
            json_file['CSF evidence'].pop(key)
    row['patient_summary'] = json.dumps(json_file, indent=4)

    # 3️⃣ Shuffle the answer options
    shuffled = CSF_OPTION_TEXTS.copy()
    rng.shuffle(shuffled)
    letters = list(string.ascii_uppercase[:len(shuffled)])
    row["options"] = "\n".join(f"{ltr}. {opt}" for ltr, opt in zip(letters, shuffled))

    # 4️⃣ Determine correct letter based on AMYLPET value
    correct_text = CSF_ANSWER_MAP[row['AMYLCSF']]
    correct_letter = letters[shuffled.index(correct_text)]
    row["ground_truth"] = correct_letter
    
    row['ground_truth_text'] = correct_text

    return row


In [89]:
# Apply the function across the PET training dataset
train_csf = train_csf.apply(add_csf_question, axis=1)

In [90]:
# Check distribution of amyloid PET labels
print(train_csf['AMYLCSF'].value_counts())
# Quick check: How many times each question variant was picked
print(train_csf['question'].value_counts())
print(train_csf['ground_truth'].value_counts())
print(train_csf['ground_truth_text'].value_counts())

AMYLCSF
1.0    400
0.0    400
Name: count, dtype: int64
question
Is the patient likely to have elevated amyloid pathology in the brain given the provided information?      166
Does the patient likely have abnormal amyloid accumulation in the brain based on the clinical data?        162
Based on the available information, is the patient likely amyloid positive in the brain?                   160
Does the evidence suggest the presence of elevated amyloid burden in the brain in this patient?            159
Is there a high likelihood of underlying amyloid pathology in the brain based on the patient's profile?    153
Name: count, dtype: int64
ground_truth
B    408
A    392
Name: count, dtype: int64
ground_truth_text
Yes    400
No     400
Name: count, dtype: int64


In [91]:
index = 10
print(train_csf.iloc[index]['options'])
print(train_csf.iloc[index]['ground_truth'])
print(train_csf.iloc[index]['ground_truth_text'])
print(train_csf.iloc[index]['AMYLCSF'])

A. No
B. Yes
A
No
0.0


In [92]:
# Select only necessary columns
train_csf = train_csf[columns]

In [93]:
train_csf['Q_TYPE'] = 'Amyloid CSF'

In [94]:
for key in csf_keys_remove:
    for i, row in train_csf.iterrows():
        if key in row['patient_summary']:
            print(i, key)
            # break

In [95]:
# Update set of NACCIDs used to avoid reusing subjects in other tasks
csf_ids = set(train_csf['ID'])
len(csf_ids)

800

## NACCETPR

In [97]:
train[(~train['ID'].isin(np_ids.union(csf_ids).union(pet_ids).union(dat_ids)))]['ETPR'].value_counts()

ETPR
NC      16468
AD      14580
FTLD     1609
LBD      1192
VD       1031
SEF       665
PSY       501
ODE       226
Name: count, dtype: int64

In [99]:
# Filter and sample patients for etiology prediction task,
# excluding those already selected for NP, PET, DAT, or MCI tasks
train_etpr_sample = (
    train
    .loc[
        (~train['ID'].isin(np_ids.union(csf_ids).union(pet_ids).union(dat_ids)))
        # (train['ETPR'] != 'ODE')  # Exclude 'other degenerative etiologies'
    ]
    .groupby('ETPR', group_keys=False)
    .apply(lambda x: x.sample(n=min(max(len(x) - 50, 0), 3000), random_state=42))  # Sample up to 1000
    .sample(frac=1, random_state=42)  # Shuffle the sampled rows
    .reset_index(drop=True)
)

# Check sampled distribution
train_etpr_sample['ETPR'].value_counts()

/scratch/ipykernel_963353/1499196860.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(max(len(x) - 50, 0), 3000), random_state=42))  # Sample up to 1000


ETPR
AD      3000
NC      3000
FTLD    1559
LBD     1142
VD       981
SEF      615
PSY      451
ODE      176
Name: count, dtype: int64

In [100]:
# Find patients not used in train_etpr_sample
remaining_patients_etpr = train[
    (~train['ID'].isin(np_ids.union(csf_ids).union(pet_ids).union(dat_ids))) &
    (~train['ID'].isin(train_etpr_sample['ID'])) & 
    # (~train['ID'].isin(train_etpr_sample_rare['ID'])) & 
    (train['NACCETPR'].isin([1]))
].reset_index(drop=True)

# Sample up to 5000 from each group
sampled_ad_nc = (
    remaining_patients_etpr
    .groupby('NACCETPR', group_keys=False)
    .apply(lambda x: x.sample(n=min(len(x), 2000), random_state=42))
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

# Check distribution
sampled_ad_nc['NACCETPR'].value_counts()

/scratch/ipykernel_963353/3635599586.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(len(x), 2000), random_state=42))


NACCETPR
1    2000
Name: count, dtype: int64

In [101]:
# Combine main etiology samples with sampled AD/NC cases
train_etpr = (
    pd.concat([train_etpr_sample, sampled_ad_nc], axis=0)
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

# Quick check
print(train_etpr['ETPR'].value_counts())
print(train_etpr['NACCUDSD'].value_counts())

ETPR
AD      5000
NC      3000
FTLD    1559
LBD     1142
VD       981
SEF      615
PSY      451
ODE      176
Name: count, dtype: int64
NACCUDSD
4    6352
3    3572
1    3000
Name: count, dtype: int64


In [102]:
import random, string

etiology_question_variants = [
    "Identify the primary etiologic diagnosis of the patient using the information provided, if applicable.",
    "Identify the primary etiology of the patient's cognitive impairment using the information provided, if applicable.",
    "Based on the clinical details, determine the main cause of the patient's cognitive impairment, if applicable.",
    "Using the information provided, identify the primary cause of the patient's cognitive decline, if applicable.",
    "Determine the primary etiology underlying the patient's cognitive impairment based on the provided information, if applicable."
]

# Raw etiology answer texts in original order
ETIOLOGY_OPTION_TEXTS = [
    "Alzheimer's disease (AD)",
    "Lewy body disease (LBD)",
    "Frontotemporal lobar degeneration and its variants, including primary progressive aphasia, corticobasal degeneration and progressive supranuclear palsy, and with or without amyotrophic lateral sclerosis (FTLD)",
    "Vascular brain injury or vascular dementia including stroke (VD)",
    "Systemic and environmental factors including infectious diseases (HIV included), metabolic, substance abuse / alcohol, medications, systemic disease and delirium (SEF)",
    "Psychiatric conditions including schizophrenia, depression, bipolar disorder, anxiety and posttraumatic stress disorder (PSY)",
    "Other (Multiple system atrophy, Essential tremor, Down syndrome, Huntington's disease, Prion disease, Traumatic brain injury, Normal-pressure hydrocephalus, Epilepsy, CNS neoplasm, etc)",
    "Not applicable (no cognitive impairment)",
]

# Mapping of ETPR code to correct answer text
ETPR_ANSWER_MAP = {
    'AD': ETIOLOGY_OPTION_TEXTS[0],
    'LBD': ETIOLOGY_OPTION_TEXTS[1],
    'FTLD': ETIOLOGY_OPTION_TEXTS[2],
    'VD': ETIOLOGY_OPTION_TEXTS[3],
    'SEF': ETIOLOGY_OPTION_TEXTS[4],
    'PSY': ETIOLOGY_OPTION_TEXTS[5],
    'ODE': ETIOLOGY_OPTION_TEXTS[6],
    'NC': ETIOLOGY_OPTION_TEXTS[7],  # fallback/default
}

def add_etpr_question(row, *, seed=None):
    rng = random.Random(seed if seed is not None else row.name)

    # 1️⃣ Randomly select a question variant
    row['question'] = rng.choice(etiology_question_variants)

    # 2️⃣ Shuffle options and assign fresh letters
    shuffled = ETIOLOGY_OPTION_TEXTS.copy()
    rng.shuffle(shuffled)
    letters = list(string.ascii_uppercase[:len(shuffled)])
    row['options'] = "\n".join(f"{ltr}. {opt}" for ltr, opt in zip(letters, shuffled))

    # 3️⃣ Get correct text from ETPR code (default to "Not applicable" if unknown)
    correct_text = ETPR_ANSWER_MAP.get(row['ETPR'])

    if correct_text is None:
        raise ValueError(f"Invalid ETPR value: {row['ETPR']}")


    # 4️⃣ Get ground truth letter from shuffled list
    if correct_text in shuffled:
        row['ground_truth'] = letters[shuffled.index(correct_text)]
    else:
        row['ground_truth'] = None  # shouldn't happen unless text mismatch
        
    row['ground_truth_text'] = correct_text

    return row


In [103]:
# Apply the function to the MCI training set
train_etpr = train_etpr.apply(add_etpr_question, axis=1)

In [104]:
print(train_etpr['ground_truth'].value_counts())
print(train_etpr['ground_truth_text'].value_counts())
print(train_etpr['question'].value_counts())
print(train_etpr['NACCUDSD'].value_counts())

ground_truth
A    1657
F    1653
B    1653
C    1607
H    1601
G    1597
E    1591
D    1565
Name: count, dtype: int64
ground_truth_text
Alzheimer's disease (AD)                                                                                                                                                                                             5000
Not applicable (no cognitive impairment)                                                                                                                                                                             3000
Frontotemporal lobar degeneration and its variants, including primary progressive aphasia, corticobasal degeneration and progressive supranuclear palsy, and with or without amyotrophic lateral sclerosis (FTLD)    1559
Lewy body disease (LBD)                                                                                                                                                                                          

In [105]:
index = 4100
print(train_etpr.iloc[index]['options'])
print(train_etpr.iloc[index]['ground_truth'])
print(train_etpr.iloc[index]['ground_truth_text'])
print(train_etpr.iloc[index]['ETPR'])

A. Vascular brain injury or vascular dementia including stroke (VD)
B. Alzheimer's disease (AD)
C. Psychiatric conditions including schizophrenia, depression, bipolar disorder, anxiety and posttraumatic stress disorder (PSY)
D. Frontotemporal lobar degeneration and its variants, including primary progressive aphasia, corticobasal degeneration and progressive supranuclear palsy, and with or without amyotrophic lateral sclerosis (FTLD)
E. Lewy body disease (LBD)
F. Not applicable (no cognitive impairment)
G. Other (Multiple system atrophy, Essential tremor, Down syndrome, Huntington's disease, Prion disease, Traumatic brain injury, Normal-pressure hydrocephalus, Epilepsy, CNS neoplasm, etc)
H. Systemic and environmental factors including infectious diseases (HIV included), metabolic, substance abuse / alcohol, medications, systemic disease and delirium (SEF)
B
Alzheimer's disease (AD)
AD


In [106]:
# Select only the columns needed for modeling
train_etpr = train_etpr[columns]
train_etpr['Q_TYPE'] = 'Primary etiology'

In [107]:
# Update set of used IDs to avoid reusing these patients elsewhere
etpr_ids = set(train_etpr['ID'])

## NACCUDSD

In [109]:
# Filter and sample patients for etiology prediction task,
# excluding those already selected for NP, PET, DAT, or MCI tasks
train_cog = (
    train
    .loc[
        (~train['ID'].isin(np_ids.union(csf_ids).union(pet_ids).union(dat_ids).union(etpr_ids)))
    ]
    .sample(frac=1, random_state=42)  # Shuffle the sampled rows
    .reset_index(drop=True)
)

# Check sampled distribution
train_cog['NACCUDSD'].value_counts()

NACCUDSD
1    13468
4     7404
3     4268
Name: count, dtype: int64

In [ ]:
# Question variants for cognitive status identification
cog_question_variants = [
    "Using the information provided, determine the patient's cognitive status.",
    "Identify the patient's cognitive status based on the given information.",
    "Based on the provided clinical details, classify the patient's cognitive status.",
    "From the information available, determine the correct cognitive status for the patient.",
    "Analyze the patient's information and identify their cognitive status."
]

# Original answer texts
COG_OPTION_TEXTS = [
    "Normal Cognition (NC)",
    "Mild Cognitive Impairment (MCI)",
    "Dementia (DE)"
]

# Define which options to show for each subset
SUBSET_OPTIONS = {
    'sub_df1': ["Normal Cognition (NC)", "Mild Cognitive Impairment (MCI)"],
    'sub_df2': ["Mild Cognitive Impairment (MCI)", "Dementia (DE)"],
    'sub_df3': ["Normal Cognition (NC)", "Dementia (DE)"]
}

In [ ]:
import json
import random
import string

def add_cog_question(df):
    """
    Process cognitive status cases by creating balanced subsets and adding question variants.
    
    Args:
        df_cog: DataFrame containing cognitive status data
    
    Returns:
        DataFrame with processed cognitive status cases including question variants
    """
    
    # Mapping for cognitive status
    mapping = {
        'Normal cognition': 'Normal Cognition (NC)',
        'MCI': 'Mild Cognitive Impairment (MCI)',
        'Dementia': 'Dementia (DE)'
    }
    
    def get_cog_stat(row):
        """Extract cognitive status from diagnosis summary"""
        try:
            diag_summary = json.loads(row['diag_summary'])
            row['COG'] = mapping[diag_summary['Clinician Diagnosis']['Cognitive status at UDS visit']]
        except (KeyError, json.JSONDecodeError, TypeError):
            # Handle cases where diag_summary might be missing or malformed
            row['COG'] = 'Unknown'
        return row
    
    # Apply cognitive status mapping
    df = df.apply(get_cog_stat, axis=1)
    
    # Get indices for each COG group
    normal_idx = df[df['COG'] == 'Normal Cognition (NC)'].index
    mci_idx = df[df['COG'] == 'Mild Cognitive Impairment (MCI)'].index
    dementia_idx = df[df['COG'] == 'Dementia (DE)'].index
    
    # Calculate 50% sample sizes for each group
    n_normal_50 = int(0.5 * len(normal_idx))
    n_mci_50 = int(0.5 * len(mci_idx))
    n_dementia_50 = int(0.5 * len(dementia_idx))
    
    # Shuffle indices for each group to ensure randomness and reproducibility
    normal_idx_shuffled = normal_idx.to_series().sample(frac=1, random_state=random_state).tolist()
    mci_idx_shuffled = mci_idx.to_series().sample(frac=1, random_state=random_state).tolist()
    dementia_idx_shuffled = dementia_idx.to_series().sample(frac=1, random_state=random_state).tolist()
    
    # Create three balanced subsets with no overlap of IDs
    
    # 1st sub dataframe: 50% Normal cognition + 50% MCI
    sub_df1_normal_idx = normal_idx_shuffled[:n_normal_50]
    sub_df1_mci_idx = mci_idx_shuffled[:n_mci_50]
    sub_df1 = pd.concat([
        df.loc[sub_df1_normal_idx],
        df.loc[sub_df1_mci_idx]
    ])


    # 2nd sub dataframe: 50% MCI + 50% Dementia
    sub_df2_mci_idx = mci_idx_shuffled[n_mci_50:]
    sub_df2_dementia_idx = dementia_idx_shuffled[:n_dementia_50]
    sub_df2 = pd.concat([
        df.loc[sub_df2_mci_idx],
        df.loc[sub_df2_dementia_idx]
    ])


    # 3rd sub dataframe: 50% Normal cognition + 50% Dementia
    sub_df3_normal_idx = normal_idx_shuffled[n_normal_50:]
    sub_df3_dementia_idx = dementia_idx_shuffled[n_dementia_50:]
    sub_df3 = pd.concat([
        df.loc[sub_df3_normal_idx],
        df.loc[sub_df3_dementia_idx]
    ])
    
    # Reset indices and shuffle each sub dataframe
    sub_df1 = sub_df1.sample(frac=1, random_state=random_state).reset_index(drop=True)
    sub_df2 = sub_df2.sample(frac=1, random_state=random_state).reset_index(drop=True)
    sub_df3 = sub_df3.sample(frac=1, random_state=random_state).reset_index(drop=True)
    
    # Verify no overlap in IDs
    assert set(sub_df1['ID']).isdisjoint(set(sub_df2['ID'])), "There are repeated IDs in sub_df1 and sub_df2"
    assert set(sub_df2['ID']).isdisjoint(set(sub_df3['ID'])), "There are repeated IDs in sub_df2 and sub_df3"
    assert set(sub_df1['ID']).isdisjoint(set(sub_df3['ID'])), "There are repeated IDs in sub_df1 and sub_df3"
    
    
    
    def add_cog_question(row, subset_name, *, seed=None):
        """Add question variants and shuffled options for cognitive status questions"""
        rng = random.Random(seed if seed is not None else row.name)
        
        # 1️⃣ Random question phrasing
        row['question'] = rng.choice(cog_question_variants)
        
        # 2️⃣ Get the relevant options for this subset
        subset_options = SUBSET_OPTIONS.get(subset_name, COG_OPTION_TEXTS)
        
        # Shuffle the answer options
        shuffled = subset_options.copy()
        rng.shuffle(shuffled)
        letters = list(string.ascii_uppercase[:len(shuffled)])
        row['options'] = "\n".join(f"{ltr}. {opt}" for ltr, opt in zip(letters, shuffled))
        
        # 3️⃣ Determine correct answer letter
        correct_text = row['COG']
        if correct_text in shuffled:
            row['ground_truth'] = letters[shuffled.index(correct_text)]
        else:
            row['ground_truth'] = None  # fallback for unrecognized codes
        
        row['ground_truth_text'] = correct_text
        
        return row
    
    # Apply question generation to each subset
    sub_df1 = sub_df1.apply(lambda row: add_cog_question(row, 'sub_df1'), axis=1).reset_index(drop=True)
    sub_df2 = sub_df2.apply(lambda row: add_cog_question(row, 'sub_df2'), axis=1).reset_index(drop=True)
    sub_df3 = sub_df3.apply(lambda row: add_cog_question(row, 'sub_df3'), axis=1).reset_index(drop=True)
    
    combined_cog_df = pd.concat([sub_df1, sub_df2, sub_df3]).sample(frac=1, random_state=random_state).reset_index(drop=True)
    combined_cog_df.drop(['COG'], inplace=True, axis=1)
    
    return combined_cog_df

In [111]:
# Apply the function to the MCI training set
train_cog = add_cog_question(train_cog)

In [112]:
print(train_cog['ground_truth'].value_counts())
print(train_cog['ground_truth_text'].value_counts())
print(train_cog['question'].value_counts())

ground_truth
B    12696
A    12444
Name: count, dtype: int64
ground_truth_text
Normal Cognition (NC)              13468
Dementia (DE)                       7404
Mild Cognitive Impairment (MCI)     4268
Name: count, dtype: int64
question
Using the information provided, determine the patient's cognitive status.                  5162
Analyze the patient's information and identify their cognitive status.                     5114
From the information available, determine the correct cognitive status for the patient.    5048
Identify the patient's cognitive status based on the given information.                    5011
Based on the provided clinical details, classify the patient's cognitive status.           4805
Name: count, dtype: int64


In [113]:
index = 10
print(train_cog.iloc[index]['options'])
print(train_cog.iloc[index]['ground_truth'])
print(train_cog.iloc[index]['ground_truth_text'])
print(train_cog.iloc[index]['NACCUDSD'])

A. Dementia (DE)
B. Normal Cognition (NC)
B
Normal Cognition (NC)
1


In [114]:
# Select only the columns needed for modeling
train_cog = train_cog[columns]

In [115]:
# Update set of used IDs to avoid reusing these patients elsewhere
cog_ids = set(train_cog['ID'])

In [116]:
train_cog['Q_TYPE'] = 'Cognitive status'

## Combine

In [117]:
train_combined = (
    pd.concat([train_np, train_csf, train_pet, train_dat, train_etpr, train_cog], axis=0)
    .sample(frac=1, random_state=42) 
    .reset_index(drop=True)
)

In [118]:
len(train_combined)

43064

In [119]:
assert len(train_combined) == len(train)

In [ ]:
question_df = train_combined.groupby(['question', 'Q_TYPE']).size().reset_index(name='count').sort_values(by=['Q_TYPE', 'count'])
# Save as CSV
csv_path = f"{directory_path}/training_data/training_data_grpo/train_question_counts.csv"
question_df.to_csv(csv_path, index=False)

In [ ]:
json.loads(train_combined.iloc[0]['patient_summary'])

In [ ]:
def remove_empty_keys(text):
    # text = text.replace("Neuropsychological battery Summary Scores", "Neuropsychological Battery Summary Scores")
    patient_file_json = json.loads(text)
    patient_file_json = {k: v for k, v in patient_file_json.items() if v}
    return json.dumps(patient_file_json)

train_combined["patient_summary"] = train_combined["patient_summary"].apply(remove_empty_keys)

In [ ]:
train_combined.head()

,ID,patient_summary,diag_summary,question,options,ground_truth,ground_truth_text,Q_TYPE
0,NACC673619,"{""Subject Demographics"": {""Subject's age at vi...","{""Clinician Judgment of Symptoms"": {""Does the ...",Identify the primary etiologic diagnosis of th...,A. Frontotemporal lobar degeneration and its v...,D,Not applicable (no cognitive impairment),Primary etiology
1,NACC588587,"{""Subject Demographics"": {""Subject's age at vi...","{""Clinician Judgment of Symptoms"": {""Does the ...","Using the provided clinical information, deter...",A. None of the above\nB. AD and LBD\nC. LBD an...,F,AD and LBD and FTLD,Neuropath
2,NACC057843,"{""Subject Demographics"": {""Subject's age at vi...","{""Clinician Judgment of Symptoms"": {""Does the ...","Based on the clinical details, determine the m...",A. Alzheimer's disease (AD)\nB. Frontotemporal...,C,Not applicable (no cognitive impairment),Primary etiology
3,NACC351044,"{""Subject Demographics"": {""Subject's age at vi...","{""Clinician Judgment of Symptoms"": {""Does the ...",Identify the primary etiologic diagnosis of th...,A. Vascular brain injury or vascular dementia ...,A,Vascular brain injury or vascular dementia inc...,Primary etiology
4,NACC001959,"{""Subject Demographics"": {""Subject's age at vi...","{""Clinician Judgment of Symptoms"": {""Does the ...",Analyze the patient's information and identify...,A. Normal Cognition (NC)\nB. Dementia (DE)\nC....,C,Mild Cognitive Impairment (MCI),Cognitive status


In [120]:
train_combined['Q_TYPE'].value_counts()

Q_TYPE
Cognitive status    25140
Primary etiology    12924
Neuropath            2000
Amyloid PET          2000
Amyloid CSF           800
DATSCAN               200
Name: count, dtype: int64

In [ ]:
train_combined.to_csv(f"{directory_path}/training_data/training_data_grpo_modified_cog_np/train_with_questions.csv", index=False)